# Module 0: Commodity Data Exploration

## Learning Objectives
By completing this notebook, you will:
1. Fetch real commodity price data from public sources
2. Explore basic properties of commodity time series
3. Identify stylized facts (volatility clustering, fat tails, seasonality)
4. Understand why standard methods fail for commodity forecasting

## Prerequisites
- pandas, matplotlib basics
- Environment setup completed

## Estimated Time: 30-45 minutes

In [ ]:
learning_objectives(["Fetch real commodity price data from public sources", "Explore basic properties of commodity time series", "Identify stylized facts (volatility clustering, fat tails, seasonality)", "Understand why standard methods fail for commodity forecasting", "pandas, matplotlib basics", "Environment setup completed"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully!")

---
## 1. Fetching Commodity Data

We'll use Yahoo Finance to get WTI Crude Oil futures data (ticker: CL=F).

In [ ]:
# Fetch WTI Crude Oil futures data
ticker = "CL=F"  # Crude Oil Futures
start_date = "2015-01-01"
end_date = "2024-12-31"

print(f"Fetching {ticker} from {start_date} to {end_date}...")
df = yf.download(ticker, start=start_date, end=end_date, progress=False)

# Calculate returns
df['Returns'] = df['Adj Close'].pct_change() * 100  # percentage returns
df = df.dropna()

print(f"\nData shape: {df.shape}")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"\nFirst few rows:")
df.head()

---
## 2. Visualize Price History

Commodity prices exhibit distinct regimes and structural breaks.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Price levels
axes[0].plot(df.index, df['Adj Close'], linewidth=1.5, color='darkblue')
axes[0].set_title('WTI Crude Oil Price (2015-2024)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($/barrel)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Annotate key events
annotations = [
    ('2020-04', 'COVID-19 crash', -40),
    ('2022-03', 'Russia-Ukraine', 120),
]
for date, label, y_pos in annotations:
    axes[0].axvline(pd.to_datetime(date), color='red', linestyle='--', alpha=0.5)
    axes[0].text(pd.to_datetime(date), y_pos, label, rotation=90, 
                 verticalalignment='bottom', fontsize=9)

# Returns
axes[1].plot(df.index, df['Returns'], linewidth=0.8, color='darkgreen', alpha=0.7)
axes[1].set_title('Daily Returns (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Return (%)', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\nObservation: Notice the regime changes and volatility clustering")

---
## 3. Stylized Fact 1: Fat Tails

Commodity returns have fatter tails than the Normal distribution (more extreme moves).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram vs Normal
axes[0].hist(df['Returns'], bins=50, density=True, alpha=0.7, 
             color='steelblue', edgecolor='black', label='Actual')

# Overlay fitted Normal
mu, sigma = df['Returns'].mean(), df['Returns'].std()
x = np.linspace(df['Returns'].min(), df['Returns'].max(), 100)
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, 
             label=f'Normal({mu:.2f}, {sigma:.2f})')
axes[0].set_title('Return Distribution vs Normal', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Daily Return (%)')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Q-Q plot
stats.probplot(df['Returns'], dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot: Returns vs Normal', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Kurtosis test
kurtosis = stats.kurtosis(df['Returns'])
print(f"\nKurtosis: {kurtosis:.2f}")
print(f"Normal distribution has kurtosis = 0")
print(f"Positive kurtosis → fatter tails (more extreme events)")
print(f"\nImplication: Normal distribution underestimates tail risk!")

---
## 4. Stylized Fact 2: Volatility Clustering

High volatility periods cluster together ("volatility comes in bunches").

In [ ]:
# Calculate rolling volatility
window = 20  # 20-day rolling window
df['Rolling_Vol'] = df['Returns'].rolling(window=window).std()

plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Rolling_Vol'], linewidth=1.5, color='purple')
plt.title(f'{window}-Day Rolling Volatility', fontsize=14, fontweight='bold')
plt.ylabel('Volatility (% std dev)', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.grid(True, alpha=0.3)
plt.axhline(df['Rolling_Vol'].mean(), color='red', linestyle='--', 
            label=f'Mean: {df["Rolling_Vol"].mean():.2f}%')
plt.legend()
plt.tight_layout()
plt.show()

# Autocorrelation of squared returns (test for volatility clustering)
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df['Returns'], lags=30, ax=axes[0], title='ACF of Returns')
plot_acf(df['Returns']**2, lags=30, ax=axes[1], title='ACF of Squared Returns')
plt.tight_layout()
plt.show()

print("\nObservation: Squared returns show strong autocorrelation")
print("This confirms volatility clustering (ARCH/GARCH effects)")

---
## 5. Stylized Fact 3: Non-Stationarity

Commodity prices exhibit regime changes and structural breaks.

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Augmented Dickey-Fuller test for stationarity
def adf_test(series, name):
    result = adfuller(series.dropna())
    print(f"\n{name}:")
    print(f"  ADF Statistic: {result[0]:.4f}")
    print(f"  p-value: {result[1]:.4f}")
    print(f"  Critical values: {result[4]}")
    if result[1] < 0.05:
        print(f"  → STATIONARY (reject unit root at 5% level)")
    else:
        print(f"  → NON-STATIONARY (fail to reject unit root)")

adf_test(df['Adj Close'], "Price Levels")
adf_test(df['Returns'], "Returns")

print("\n" + "="*60)
print("Implication: Model returns (stationary), not prices directly")
print("="*60)

---
## 6. Summary Statistics

In [ ]:
print("WTI CRUDE OIL SUMMARY STATISTICS")
print("="*50)

summary = {
    'Mean Return (% per day)': df['Returns'].mean(),
    'Std Dev (% per day)': df['Returns'].std(),
    'Annualized Volatility (%)': df['Returns'].std() * np.sqrt(252),
    'Skewness': stats.skew(df['Returns']),
    'Kurtosis': stats.kurtosis(df['Returns']),
    'Min Return (%)': df['Returns'].min(),
    'Max Return (%)': df['Returns'].max(),
    'Sharpe Ratio (ann.)': (df['Returns'].mean() * 252) / (df['Returns'].std() * np.sqrt(252)),
}

for key, value in summary.items():
    print(f"{key:.<40} {value:>8.3f}")

print("\n" + "="*50)
print("KEY TAKEAWAYS:")
print("="*50)
print("1. Fat tails (kurtosis > 0) → Need robust models")
print("2. Volatility clustering → GARCH or stochastic volatility")
print("3. Non-stationary prices → Model returns or use state space")
print("4. Regime changes visible → Consider Markov-switching models")
print("\nThese properties motivate Bayesian time series methods!")

---
## 7. Exercise: Try Another Commodity

Explore a different commodity to see if these stylized facts hold.

**Suggested tickers:**
- Natural Gas: NG=F
- Gold: GC=F
- Corn: ZC=F
- Copper: HG=F

In [ ]:
# Your code here - pick a different commodity
your_ticker = "NG=F"  # Change this

df_new = yf.download(your_ticker, start=start_date, end=end_date, progress=False)
df_new['Returns'] = df_new['Adj Close'].pct_change() * 100
df_new = df_new.dropna()

# Quick plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_new.index, df_new['Adj Close'])
ax.set_title(f'{your_ticker} Price History', fontsize=14, fontweight='bold')
ax.set_ylabel('Price')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nKurtosis: {stats.kurtosis(df_new['Returns']):.2f}")
print(f"Ann. Volatility: {df_new['Returns'].std() * np.sqrt(252):.1f}%")

---
## Completion Checklist

- [ ] Successfully fetched commodity data
- [ ] Observed fat tails in return distribution
- [ ] Identified volatility clustering
- [ ] Confirmed non-stationarity of price levels
- [ ] Explored a second commodity

## Next Steps

These properties motivate the Bayesian methods we'll learn:
- **Module 1:** Bayesian framework for uncertainty quantification
- **Module 3:** State space models for non-stationary series
- **Module 7:** Regime-switching for structural breaks

Proceed to Module 1 after completing the diagnostic quiz!